# 07 Final results: report tables, hypothesis verdicts, error analysis, report figures and a run check of every notebook

Writes `results/final_tables.md` and `results/report_figures.md`. Needs the FMA data (it refits the random forest to get its test predictions); the CNN predictions come from `results/spec/cnn_test_predictions.csv`.

In [1]:
import sys; sys.path.append('..')
import json
import numpy as np
import pandas as pd
import nbformat
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.config import FIGURES, RESULTS, ROOT, SEED
from src.data import load_fma, fma_xy
from src.evaluate import evaluate, results_table

names = {"zeror": "ZeroR", "naive_bayes": "Gaussian NB", "logistic_regression": "Logistic regression",
         "decision_tree": "Decision tree", "knn": "KNN", "random_forest": "Random forest", "mlp": "MLP",
         "logreg_summary": "Logistic regression (summary features)", "cnn": "CNN"}
cols = {"accuracy": "Accuracy", "macro_precision": "Macro-P", "macro_recall": "Macro-R",
        "macro_f1": "Macro-F1", "mcc": "MCC", "macro_auc_ovr": "Macro-AUC"}
tables = {p: results_table(p)[list(cols)].rename(index=names, columns=cols).rename_axis("Model") for p in ("fma", "spec")}
clustering = pd.DataFrame([json.loads((RESULTS / "fma_clustering" / "kmeans_k8.json").read_text())["metrics"]],
                          index=pd.Index(["k-means, k = 8"], name="Model"))[["ari", "nmi", "silhouette"]].rename(
                          columns={"ari": "ARI", "nmi": "NMI", "silhouette": "Silhouette"})
tables["fma"]

,Accuracy,Macro-P,Macro-R,Macro-F1,MCC,Macro-AUC
Model,,,,,,
Random forest,0.4900,0.4872,0.4900,0.4867,0.4175,0.8280
MLP,0.4650,0.4615,0.4650,0.4625,0.3888,0.8111
Logistic regression,0.4512,0.4494,0.4512,0.4494,0.3731,0.8131
Decision tree,0.3625,0.3626,0.3625,0.3598,0.2719,0.7398
Gaussian NB,0.3713,0.3867,0.3712,0.3494,0.2866,0.7810
KNN,0.2975,0.3080,0.2975,0.2938,0.1988,0.6660
ZeroR,0.1250,0.0156,0.1250,0.0278,0.0000,0.5000


In [2]:
tables["spec"]

,Accuracy,Macro-P,Macro-R,Macro-F1,MCC,Macro-AUC
Model,,,,,,
CNN,0.8520,0.8596,0.8520,0.8516,0.824,0.9844
Logistic regression (summary features),0.7473,0.7448,0.7473,0.7454,0.697,0.9403
ZeroR,0.1668,0.0278,0.1667,0.0477,0.000,0.5000


## 1. Per-genre recall of the two best models

The random forest is refitted from its saved hyperparameters with the same seed, which reproduces its saved test scores exactly; the CNN's test predictions were saved by notebook 06.

In [3]:
fma = load_fma()
X_train, y_train = fma_xy(fma, "training")
X_test, y_test = fma_xy(fma, "test")
rf_saved = json.loads((RESULTS / "fma" / "random_forest.json").read_text())
rf = Pipeline([("scale", StandardScaler()),
               ("model", RandomForestClassifier(**rf_saved["params"], random_state=SEED, n_jobs=-1))]).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_metrics, rf_class = evaluate(y_test, rf_pred, labels=rf.classes_)
assert rf_metrics["macro_f1"] == rf_saved["metrics"]["macro_f1"], "refit does not reproduce the saved random forest"

cnn = pd.read_csv(RESULTS / "spec" / "cnn_test_predictions.csv")
_, cnn_class = evaluate(cnn["true"], cnn["pred"])
recall = {"fma": rf_class["recall_tpr"].sort_values().rename("Recall").rename_axis("Genre").to_frame(),
          "spec": cnn_class["recall_tpr"].sort_values().rename("Recall").rename_axis("Genre").to_frame()}
pd.concat(recall, axis=1)

,fma,spec
,Recall,Recall
Genre,,
Folk,0.24,0.9149
Experimental,0.31,NaN
Pop,0.42,NaN
Instrumental,0.46,NaN
International,0.51,NaN
Electronic,0.62,0.7788
Rock,0.67,0.9101
Hip-Hop,0.69,0.7954


## 2. Hypothesis verdicts

In [4]:
fma_f1, spec_f1 = results_table("fma")["macro_f1"], results_table("spec")["macro_f1"]
classical = fma_f1.drop(["mlp", "zeror"])
lowest = recall["fma"]["Recall"].head(3)
predicted = {"Experimental", "International", "Pop"}
hit = len(predicted & set(lowest.index))
gain_mlp = fma_f1["mlp"] - classical.max()
gain_cnn = spec_f1["cnn"] - spec_f1["logreg_summary"]

verdicts = {
    "H1": (["Not supported", "Partly supported", "Partly supported", "Supported"][hit] +
           f". The random forest's lowest-recall genres are " + ", ".join(f"{g} ({r:.2f})" for g, r in lowest.items()) +
           f"; {hit} of the 3 predicted genres (Experimental, International, Pop) are among them, and International reaches "
           f"{recall['fma'].loc['International', 'Recall']:.2f}."),
    # H2 predicts at most a small gain, so it fails only if the MLP beats the best classical model by a clear margin.
    "H2": (("Supported" if gain_mlp < 0.03 else "Not supported") +
           f". The MLP reached {fma_f1['mlp']:.3f} test macro-F1 against {classical.max():.3f} for the "
           f"{names[classical.idxmax()].lower()}, a difference of {gain_mlp:+.3f}."),
    "H3": (("Supported" if gain_cnn > 0 else "Not supported") +
           f". The CNN reached {spec_f1['cnn']:.3f} test macro-F1 against {spec_f1['logreg_summary']:.3f} for logistic "
           f"regression on per-band summary features, a difference of {gain_cnn:+.3f}."),
}
for h, v in verdicts.items():
    print(f"{h}: {v}\n")

H1: Partly supported. The random forest's lowest-recall genres are Folk (0.24), Experimental (0.31), Pop (0.42); 2 of the 3 predicted genres (Experimental, International, Pop) are among them, and International reaches 0.51.

H2: Supported. The MLP reached 0.463 test macro-F1 against 0.487 for the random forest, a difference of -0.024.

H3: Supported. The CNN reached 0.852 test macro-F1 against 0.745 for logistic regression on per-band summary features, a difference of +0.106.



## 3. Error analysis: most confused genre pairs

Counts in both directions on the test split, so A ↔ B adds A predicted as B and B predicted as A.

In [5]:
def confused_pairs(y_true, y_pred, n=3):
    labels = sorted(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows = [(f"{a} ↔ {b}", cm[i, j] + cm[j, i], f"{a} → {b}: {cm[i, j]}, {b} → {a}: {cm[j, i]}")
            for i, a in enumerate(labels) for j, b in enumerate(labels) if i < j]
    return pd.DataFrame(rows, columns=["Pair", "Errors", "By direction"]).nlargest(n, "Errors").set_index("Pair")

pairs = {"fma": confused_pairs(y_test.to_numpy(), rf_pred), "spec": confused_pairs(cnn["true"].to_numpy(), cnn["pred"].to_numpy())}
pd.concat(pairs, names=["Pipeline"])

Errors  \
Pipeline Pair                           
fma      Folk ↔ Instrumental       48   
         Experimental ↔ Folk       41   
         Folk ↔ International      35   
spec     Folk ↔ Jazz               78   
         Electronic ↔ Jazz         44   
         Folk ↔ Rock               41   

                                                                    By direction  
Pipeline Pair                                                                     
fma      Folk ↔ Instrumental    Folk → Instrumental: 15, Instrumental → Folk: 33  
         Experimental ↔ Folk    Experimental → Folk: 14, Folk → Experimental: 27  
         Folk ↔ International  Folk → International: 17, International → Folk...  
spec     Folk ↔ Jazz                            Folk → Jazz: 10, Jazz → Folk: 68  
         Electronic ↔ Jazz           Electronic → Jazz: 39, Jazz → Electronic: 5  
         Folk ↔ Rock                            Folk → Rock: 23, Rock → Folk: 18

**FMA (random forest).** All three pairs involve Folk: Folk ↔ Instrumental (48 errors), Experimental ↔ Folk (41) and Folk ↔ International (35). Notebook 02 already showed this structure without labels: k-means put Folk and Instrumental in the same two clusters (2 and 4) and split Experimental across clusters shared with Instrumental and Rock. Notebook 01 explains why: the first two principal components hold only 19% of the variance and the genres overlap heavily, and Experimental and International have the weakest individual features (max |r| 0.18). These genres are defined partly by culture and intent rather than sound, so acoustic descriptors cannot fully separate them.

**Spectrograms (CNN).** Jazz is in two of the three pairs: Folk ↔ Jazz (78 errors, 68 of them Jazz predicted as Folk) and Electronic ↔ Jazz (44, mostly Electronic predicted as Jazz), followed by Folk ↔ Rock (41). Notebook 05's mean spectrograms show why: Jazz and Folk are the two genres closest to the all-genre average, so their spectral shape differs least, and Jazz was also the baseline's weakest genre (recall 0.58). The CNN raised Jazz recall to 0.73 by using time-frequency patterns, but the Jazz → Folk confusion remains its largest error.

## 4. Figures for the report (at most 6)

In [6]:
figures = [
    ("fma_kmeans_pca.png", "R2", "FMA training tracks on the first two principal components, coloured by k-means cluster (k = 8, left) and by genre (right). Genres overlap heavily and the clusters do not follow them (ARI 0.081, NMI 0.133)."),
    ("spec_mean_per_genre.png", "R2", "Mean mel-spectrogram per genre over 300 training clips (top) and its difference from the all-genre mean (bottom). Genres differ mainly in average spectral shape, for example Rock louder in almost every band and Classical quieter in the lowest bands."),
    ("fma_baselines_validation_curves.png", "R3", "Training and 5-fold CV macro-F1 across each classical model's main hyperparameter; the dashed line marks the chosen value. The gap between the curves shows over- or underfitting."),
    ("fma_baselines_confusion.png", "R3", "Row-normalised test confusion matrix of the random forest, the best FMA model (0.487 macro-F1). Folk, Experimental and Pop have the lowest recall."),
    ("fma_mlp_loss_curves.png", "R4", "MLP training and validation loss. Left: the unregularised (512, 256) network overfits from the first epoch. Right: the selected (256) network at learning rate 1e-4 converges with a much smaller gap."),
    ("spec_cnn_confusion.png", "R4", "Row-normalised test confusion matrix of the selected CNN (4 blocks with dropout, 0.852 macro-F1). Jazz predicted as Folk is the largest error."),
]
missing = [f for f, _, _ in figures if not (FIGURES / f).exists()]
assert not missing, f"missing figures: {missing}"
lines = ["# Report figures", "", "Generated by `notebooks/07_results.ipynb`. In report order; captions are drafts.", ""]
lines += [f"{i}. **Figure {i}** (`figures/{f}`, {req}): {cap}" for i, (f, req, cap) in enumerate(figures, 1)]
(RESULTS / "report_figures.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

# Report figures

Generated by `notebooks/07_results.ipynb`. In report order; captions are drafts.

1. **Figure 1** (`figures/fma_kmeans_pca.png`, R2): FMA training tracks on the first two principal components, coloured by k-means cluster (k = 8, left) and by genre (right). Genres overlap heavily and the clusters do not follow them (ARI 0.081, NMI 0.133).
2. **Figure 2** (`figures/spec_mean_per_genre.png`, R2): Mean mel-spectrogram per genre over 300 training clips (top) and its difference from the all-genre mean (bottom). Genres differ mainly in average spectral shape, for example Rock louder in almost every band and Classical quieter in the lowest bands.
3. **Figure 3** (`figures/fma_baselines_validation_curves.png`, R3): Training and 5-fold CV macro-F1 across each classical model's main hyperparameter; the dashed line marks the chosen value. The gap between the curves shows over- or underfitting.
4. **Figure 4** (`figures/fma_baselines_confusion.png`, R3): Row-normalised test confus

## 5. Did every notebook run top to bottom?

A notebook passes if every code cell has been executed, in order, with consecutive execution counts and no error outputs.

In [7]:
def check(path):
    cells = [c for c in nbformat.read(path, as_version=4).cells if c.cell_type == "code"]
    counts = [c.get("execution_count") for c in cells]
    errors = sum(o.get("output_type") == "error" for c in cells for o in c.get("outputs", []))
    in_order = None not in counts and all(b == a + 1 for a, b in zip(counts, counts[1:]))
    return {"code cells": len(cells), "first count": counts[0], "errors": errors, "passes": in_order and not errors}

runs = pd.DataFrame({p.name: check(p) for p in sorted((ROOT / "notebooks").glob("0[0-6]_*.ipynb"))}).T
print("Notebooks that fail:", list(runs.index[~runs["passes"].astype(bool)]) or "none")
runs

Notebooks that fail: none


,code cells,first count,errors,passes
00_quickstart.ipynb,5,1,0,True
01_fma_eda.ipynb,11,1,0,True
02_fma_clustering.ipynb,5,1,0,True
03_fma_baselines.ipynb,9,1,0,True
04_fma_mlp.ipynb,8,1,0,True
05_spec_eda_baseline.ipynb,8,1,0,True
06_spec_cnn.ipynb,10,4,0,True


## 6. Write `results/final_tables.md`

In [8]:
def md_table(df):
    fmt = lambda v: f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)
    head = [df.index.name or ""] + list(df.columns)
    rows = [[str(i)] + [fmt(v) for v in r] for i, r in zip(df.index, df.itertuples(index=False))]
    return "\n".join("| " + " | ".join(r) + " |" for r in [head, ["---"] * len(head), *rows])

sections = [
    ("Table 1. Pipeline A (FMA, 8 genres): test results", tables["fma"]),
    ("Table 2. Pipeline B (mel-spectrograms, 6 genres): test results", tables["spec"]),
    ("Table 3. k-means (k = 8) against FMA genres, training split", clustering),
    ("Table 4. Per-genre test recall: random forest (FMA)", recall["fma"]),
    ("Table 5. Per-genre test recall: CNN (spectrograms)", recall["spec"]),
    ("Table 6. Most confused genre pairs: random forest (FMA)", pairs["fma"]),
    ("Table 7. Most confused genre pairs: CNN (spectrograms)", pairs["spec"]),
]
text = ["# Final results", "", "Generated by `notebooks/07_results.ipynb`. Test split unless stated. "
        "Scores are comparable only within a pipeline: the datasets differ in genres, size and clip length.", ""]
for title, df in sections:
    text += [f"## {title}", "", md_table(df), ""]
text += ["## Hypotheses", ""] + [f"- **{h}:** {v}" for h, v in verdicts.items()]
(RESULTS / "final_tables.md").write_text("\n".join(text) + "\n", encoding="utf-8")
print("\n".join(text))

# Final results

Generated by `notebooks/07_results.ipynb`. Test split unless stated. Scores are comparable only within a pipeline: the datasets differ in genres, size and clip length.

## Table 1. Pipeline A (FMA, 8 genres): test results

| Model | Accuracy | Macro-P | Macro-R | Macro-F1 | MCC | Macro-AUC |
| --- | --- | --- | --- | --- | --- | --- |
| Random forest | 0.490 | 0.487 | 0.490 | 0.487 | 0.417 | 0.828 |
| MLP | 0.465 | 0.462 | 0.465 | 0.463 | 0.389 | 0.811 |
| Logistic regression | 0.451 | 0.449 | 0.451 | 0.449 | 0.373 | 0.813 |
| Decision tree | 0.362 | 0.363 | 0.362 | 0.360 | 0.272 | 0.740 |
| Gaussian NB | 0.371 | 0.387 | 0.371 | 0.349 | 0.287 | 0.781 |
| KNN | 0.297 | 0.308 | 0.297 | 0.294 | 0.199 | 0.666 |
| ZeroR | 0.125 | 0.016 | 0.125 | 0.028 | 0.000 | 0.500 |

## Table 2. Pipeline B (mel-spectrograms, 6 genres): test results

| Model | Accuracy | Macro-P | Macro-R | Macro-F1 | MCC | Macro-AUC |
| --- | --- | --- | --- | --- | --- | --- |
| CNN | 0.852 | 0.860 | 0.

## Findings

- **Best models:** the random forest is the best FMA model (0.487 test macro-F1, 0.417 MCC, 0.828 macro-AUC), ahead of the MLP (0.463) and logistic regression (0.449); every model clearly beats ZeroR (0.028). On the spectrograms the CNN (0.852 macro-F1, 0.824 MCC, 0.984 macro-AUC) beats logistic regression on summary features (0.745).
- **H1 (lowest recall: Experimental, International, Pop):** partly supported. The random forest's lowest-recall genres are Folk (0.24), Experimental (0.31) and Pop (0.42); International reaches 0.51, so Folk replaces it. The MLP has the same bottom three.
- **H2 (any MLP gain over classical models is small):** supported. The MLP finished 0.024 macro-F1 below the random forest, so it gave no gain on these 518 tabular features.
- **H3 (CNN beats the summary-feature baseline):** supported. The CNN is 0.106 macro-F1 higher, about 270 more of the 2,608 test clips, because it keeps time-frequency patterns that per-band averages lose.
- **Error analysis:** on FMA every top confusion involves Folk (Folk ↔ Instrumental 48, Experimental ↔ Folk 41, Folk ↔ International 35), the same overlap that k-means and PCA showed without labels. On the spectrograms Jazz dominates the errors (Folk ↔ Jazz 78, mostly Jazz predicted as Folk; Electronic ↔ Jazz 44), matching the mean-spectrogram finding that Jazz and Folk are closest to the average spectral shape.
- **Reproducibility:** notebooks 00–06 all ran top to bottom without errors. Notebook 06 ran on a Kaggle T4 GPU; its run counts start at 4 because three cells were tested first, but every cell ran in order in one session.